# Fase 3 - Revisión humana de Matriz Madre

Este notebook es la interfaz human-in-the-loop. La lógica reusable vive en `src/`; aquí se revisan crosswalks, territorios, variables de baja cobertura y decisiones metodológicas antes de cerrar Fase 3.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PHASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PHASE_DIR))
OUT = PHASE_DIR / 'outputs'
CONFIG = PHASE_DIR / 'config' / 'fase3'
OUT, CONFIG

## 1. Crosswalk Microsoft

Revisar que cada país Microsoft tenga `action` aprobado/corregido por humano. Las filas pendientes no entran a la matriz final.

In [ ]:
crosswalk = pd.read_csv(OUT / 'fase3_geo_crosswalk_manual.csv')
display(crosswalk.sort_values(['action', 'confidence_score']).head(30))
crosswalk['action'].value_counts(dropna=False)

## 2. Territorios y entidades excluidas

Confirmar que regiones, agregados, organizaciones y territorios no aprobados queden fuera de `matriz_madre_wide.csv` pero documentados para auditoría.

In [ ]:
universe = pd.read_csv(OUT / 'fase3_universo_geografico.csv')
excluded_geo = pd.read_csv(OUT / 'fase3_entidades_excluidas_geografia.csv')
display(universe[universe['iso3'].isin(['HKG','PRI','TWN','XKX','MAC','AFE','AFW','ANT'])])
display(excluded_geo.head(30))

## 3. Variables de baja cobertura y redundancias

Estas variables son reales, pero Fase 4 debe tratarlas con cautela. Revisar `fase4_role`, `included_in_fase4_eda`, `is_primary` y `redundant_with`.

In [ ]:
dictionary = pd.read_csv(OUT / 'fase3_diccionario_variables.csv')
display(dictionary[dictionary['pct_complete'] < 30][['variable_matriz','source_id','bloque_tematico','pct_complete','fase4_role','known_limitations']].head(50))
display(dictionary[dictionary['redundant_with'].fillna('').ne('')][['variable_matriz','source_id','is_primary','redundant_with','fase4_role']])

## 4. Chile y cobertura por bloque

Verificar que Chile tenga cobertura en los bloques necesarios para alimentar Fase 4.

In [ ]:
snapshot = pd.read_csv(OUT / 'matriz_larga_snapshot.csv')
chile = snapshot[snapshot['iso3'].eq('CHL')].merge(dictionary[['variable_matriz','bloque_tematico','fase4_role']], on='variable_matriz', how='left')
display(chile['bloque_tematico'].value_counts())
display(chile[['source_id','variable_matriz','year_used','value_original','confidence_level','bloque_tematico','fase4_role']].head(80))

## 5. Registrar revisión

Si se realizan correcciones manuales, actualizar los CSV/YAML correspondientes y volver a ejecutar `python -m src.fase3 build-all` desde la raíz `FASE3`.